# Datheon Internship — Task 4
## NLP Sentiment Classification on Customer Reviews (TF-IDF)

**Objective:** Clean review text, transform it into TF-IDF features, train Multinomial Naive Bayes and Logistic Regression, compare accuracy and other classification metrics, and test custom review phrases.

### Deliverables
- Dataset loading and validation
- HTML/punctuation cleanup, lowercasing, stopword removal, and optional lemmatization
- `TfidfVectorizer(max_features=5000)`
- Multinomial Naive Bayes and Logistic Regression
- Accuracy, precision, recall, F1, confusion matrix, and classification report
- Live `predict_sentiment()` helper

**Dataset note:** The supplied CSV URL may be unavailable or may not have the expected schema. The notebook first attempts it, then falls back to the NLTK movie reviews corpus (downloaded when available). If neither source is accessible, it raises a clear error rather than fabricating data.


## 1. Install dependencies if needed

In [ ]:
# Run in a notebook cell or terminal if required:
# %pip install pandas numpy matplotlib seaborn scikit-learn nltk

## 2. Import libraries

In [ ]:
import re
import html
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
print("Imports complete.")

## 3. Load a review dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/datasets/sample-reviews/main/sentiment_reviews.csv"

def load_reviews():
    try:
        candidate = pd.read_csv(DATA_URL)
        print("Loaded the provided CSV.")
        return candidate, "provided CSV"
    except Exception as exc:
        print(f"Provided CSV unavailable or unreadable: {exc}")

    try:
        import nltk
        from nltk.corpus import movie_reviews
        try:
            movie_reviews.categories()
        except LookupError:
            nltk.download("movie_reviews", quiet=True)
        try:
            nltk.data.find("corpora/movie_reviews")
        except LookupError:
            nltk.download("movie_reviews", quiet=True)

        records = []
        for label in movie_reviews.categories():
            for fileid in movie_reviews.fileids(label):
                records.append({
                    "text": " ".join(movie_reviews.words(fileid)),
                    "sentiment": label
                })
        if records:
            print("Loaded NLTK movie_reviews corpus as fallback.")
            return pd.DataFrame(records), "NLTK movie_reviews fallback"
    except Exception as exc:
        print(f"NLTK fallback unavailable: {exc}")

    raise RuntimeError(
        "Could not load a review dataset. Download a labeled CSV with review text "
        "and sentiment/label columns, then set DATA_URL or load it from a local path."
    )

raw_df, dataset_source = load_reviews()
print("Dataset source:", dataset_source)
print("Shape:", raw_df.shape)
display(raw_df.head())

## 4. Identify text and sentiment columns

In [ ]:
def find_column(columns, candidates):
    lower_map = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return None

text_col = find_column(raw_df.columns, ["text", "review", "review_text", "content", "sentence", "comment"])
label_col = find_column(raw_df.columns, ["sentiment", "label", "polarity", "target", "class"])

if text_col is None or label_col is None:
    raise ValueError(
        f"Could not infer text/label columns from {list(raw_df.columns)}. "
        "Set text_col and label_col manually for this dataset."
    )

df = raw_df[[text_col, label_col]].rename(columns={text_col: "text", label_col: "sentiment"}).copy()
df = df.dropna(subset=["text", "sentiment"])
df["text"] = df["text"].astype(str)
df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()
print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", sorted(df["sentiment"].unique()))
display(df["sentiment"].value_counts().to_frame("count"))

## 5. Normalize labels and clean text

In [ ]:
# Convert common positive/negative label variants into a consistent binary target.
positive_labels = {"positive", "pos", "1", "1.0", "positive sentiment"}
negative_labels = {"negative", "neg", "0", "0.0", "negative sentiment"}

def normalize_label(value):
    value = str(value).strip().lower()
    if value in positive_labels:
        return "positive"
    if value in negative_labels:
        return "negative"
    return np.nan

df["sentiment"] = df["sentiment"].map(normalize_label)
df = df.dropna(subset=["sentiment"]).copy()

if df["sentiment"].nunique() != 2:
    raise ValueError(
        "Expected binary positive/negative labels. Inspect the dataset labels and "
        "update normalize_label() for the source's label convention."
    )

def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)        # HTML tags
    text = re.sub(r"https?://\\S+|www\\.\\S+", " ", text)  # URLs
    text = re.sub(r"[^A-Za-z\\s']", " ", text) # punctuation/numbers
    text = text.lower()
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].map(clean_text)
df = df[df["clean_text"].str.len() > 0].copy()
display(df[["text", "clean_text", "sentiment"]].head())

## 6. Split the data

Use a stratified 80/20 split to preserve the positive/negative proportions. Text vectorization is performed inside the pipeline, so the vocabulary and IDF weights are learned from training text only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.20, random_state=RANDOM_STATE, stratify=df["sentiment"]
)
print("Train samples:", len(X_train))
print("Test samples:", len(X_test))
print("Training label distribution:")
display(y_train.value_counts(normalize=True).to_frame("proportion"))

## 7. Train TF-IDF pipelines

In [ ]:
# Scikit-learn's built-in English stopword list avoids extra downloads.
# Note: stopword removal can remove negation words like "not"; the next cell
# removes those negation terms from the stopword set to preserve sentiment cues.
stop_words = set(ENGLISH_STOP_WORDS) - {"no", "nor", "not", "never"}

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words=sorted(stop_words), ngram_range=(1, 2), min_df=2)),
    ("classifier", MultinomialNB())
])
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words=sorted(stop_words), ngram_range=(1, 2), min_df=2)),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
])

models = {
    "Multinomial Naive Bayes": nb_pipeline,
    "Logistic Regression": lr_pipeline
}
fitted_models = {}
predictions = {}
probabilities = {}
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    pred = model.predict(X_test)
    predictions[name] = pred
    prob = model.predict_proba(X_test)[:, list(model.classes_).index("positive")]
    probabilities[name] = prob
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(rows).set_index("Model").sort_values("F1-score (macro)", ascending=False)
display(results_df.style.format("{:.4f}"))

## 8. Classification reports

In [ ]:
for name, pred in predictions.items():
    print("=" * 65)
    print(name)
    print(classification_report(y_test, pred, labels=["negative", "positive"], zero_division=0))

## 9. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred, labels=["negative", "positive"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 10. Accuracy and F1 comparison

In [ ]:
plot_df = results_df.reset_index().melt(
    id_vars="Model", var_name="Metric", value_name="Score"
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1.05)
plt.title("Sentiment Classifier Benchmark")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 11. Predict sentiment for custom text

In [ ]:
def predict_sentiment(text, model_name=None):
    \"\"\"Predict positive/negative sentiment for one custom review phrase.

    Args:
        text: User-supplied review text.
        model_name: Optional model key. Defaults to the model with the highest
                    macro F1-score on this held-out split.
    Returns:
        Dictionary with model, predicted sentiment, and positive-class probability.
    \"\"\"
    if model_name is None:
        model_name = results_df.index[0]
    if model_name not in fitted_models:
        raise ValueError(f"Choose one of: {list(fitted_models)}")

    cleaned = clean_text(text)
    model = fitted_models[model_name]
    predicted = model.predict([cleaned])[0]
    class_index = list(model.classes_).index("positive")
    positive_probability = float(model.predict_proba([cleaned])[0, class_index])
    return {
        "model": model_name,
        "sentiment": predicted,
        "positive_probability": round(positive_probability, 4)
    }

# Try your own phrase by editing the string below.
examples = [
    "The product is excellent and works perfectly.",
    "Very disappointing quality, I would not recommend it.",
    "The movie was not bad, but the story was slow."
]
for phrase in examples:
    print(phrase)
    print(predict_sentiment(phrase))
    print()

## 12. Optional: Inspect most informative Logistic Regression terms

In [ ]:
lr = fitted_models["Logistic Regression"]
vectorizer = lr.named_steps["tfidf"]
classifier = lr.named_steps["classifier"]
feature_names = np.array(vectorizer.get_feature_names_out())

if len(classifier.classes_) == 2:
    coef = classifier.coef_[0]
    positive_idx = np.argsort(coef)[-10:][::-1]
    negative_idx = np.argsort(coef)[:10]

    print("Terms most associated with positive sentiment:")
    display(pd.DataFrame({"term": feature_names[positive_idx], "coefficient": coef[positive_idx]}))

    print("Terms most associated with negative sentiment:")
    display(pd.DataFrame({"term": feature_names[negative_idx], "coefficient": coef[negative_idx]}))

## 13. Conclusion and reporting checklist

- Report measured accuracy, precision, recall, and F1 from the held-out test set.
- Explain why TF-IDF converts text into weighted numeric features.
- Compare Naive Bayes and Logistic Regression using the same split.
- Include the confusion matrix and a few custom `predict_sentiment()` examples.
- Avoid claiming general performance beyond the dataset tested.

### Suggested GitHub structure
```text
datheon-task-4-sentiment-classification/
├── sentiment_tfidf_classifier.ipynb
├── README.md
└── requirements.txt
```

**Preprocessing note:** This notebook uses regex normalization and Scikit-Learn's English stopword list. If your evaluator strictly requires lemmatization, you can add NLTK WordNetLemmatizer after downloading the `wordnet` and `omw-1.4` resources. Keep negation terms such as "not" because they are important for sentiment.
